# 03 - Model B trajectories (cross-model transfer)

**Session settings:** Accelerator `GPU T4 x2`, Internet **on**, Secret `tracewarden-token-huggingface` ticked
for this notebook. **Input:** dataset `tw-src-1`.

Model A was Qwen2.5-7B-Instruct-AWQ. Model B here is **Qwen2.5-14B-Instruct-AWQ**: same tool-calling contract
(hermes parser, parallel calls supported), different scale.

Llama-3.1-8B-Instruct-AWQ-INT4 was tried first and **does not work with AgentDojo**: its chat template rejects
parallel tool calls (`This model only supports single tool-calls at once!`), which AgentDojo agents emit
constantly. Neither `llama3_json` nor `pythonic` avoids it. Record this as a documented limitation.

Everything writes under the tag `qwen14b`, so model A's data cannot be overwritten.

In [ ]:
# 1 - install and locate the tracewarden source
!pip -q install vllm agentdojo openai 2>&1 | tail -1

import glob, sys, os
cand = glob.glob("/kaggle/input/**/tw-src-1/**/tracewarden/__init__.py", recursive=True)
assert cand, "attach the tw-src-1 dataset (Add Input -> Your Work)"
SRC = os.path.dirname(os.path.dirname(cand[0]))
sys.path.insert(0, SRC)
import tracewarden
print("tracewarden", tracewarden.__version__, "from", SRC)

MODEL  = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
PARSER = 'hermes'
TAG    = 'qwen14b'
SUITES = ['banking', 'slack', 'travel']
WORK   = '/kaggle/working'

In [ ]:
# 2 - start vLLM. 14B AWQ is ~9 GB of weights: use both T4s for KV-cache headroom.
try:
    srv.terminate(); srv.wait()
except NameError:
    pass
import subprocess, time, requests
srv = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--quantization', 'awq', '--dtype', 'half',
     '--max-model-len', '8192', '--gpu-memory-utilization', '0.90',
     '--tensor-parallel-size', '2',
     '--enable-auto-tool-choice', '--tool-call-parser', PARSER, '--port', '8000'],
    stdout=open('vllm_b.log', 'w'), stderr=subprocess.STDOUT)

for i in range(180):
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
            print("server up after", i * 10, "s"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print(open('vllm_b.log').read()[-3000:])

# If it OOMs: drop --tensor-parallel-size, or --max-model-len 4096.
# If the download stalls, check: !tail -5 vllm_b.log

In [ ]:
# 3 - point AgentDojo at the local server
import os
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY']  = 'sk-none'
os.environ['TW_LLM_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['TW_LLM_MODEL']    = MODEL

BASE = f'python -m agentdojo.scripts.benchmark --model VLLM_PARSED --model-id {MODEL}'

from openai import OpenAI
print(requests.get('http://localhost:8000/v1/models').json()['data'][0]['id'])
print(OpenAI().chat.completions.create(model=MODEL, max_tokens=5,
      messages=[{"role": "user", "content": "Say OK."}]).choices[0].message.content)

In [ ]:
# 4 - helpers: convert logs to labeled trajectories, and upload
from collections import Counter
from tracewarden.io.agentdojo import convert_dir
from tracewarden.schema import save_jsonl

def convert(logdir, out_name, attacked):
    """attacked=True keeps only runs that had an attack (drops AgentDojo's none/none calibration runs)."""
    trajs = convert_dir(logdir, source=f"agentdojo-{TAG}")
    trajs = [t for t in trajs if bool(t.meta.get("attack")) == attacked]
    save_jsonl(trajs, f"{WORK}/{out_name}.jsonl")
    print(f"{out_name}: {len(trajs)} trajectories", dict(Counter(t.category for t in trajs)),
          "| steps:", dict(Counter(s.label for t in trajs for s in t.steps)),
          "| oracle disagreements:", sum(1 for t in trajs if t.meta.get("oracle_disagrees")))
    return trajs

def upload(patterns=('logs_*/**', '*.jsonl')):
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi, login
    login(UserSecretsClient().get_secret("tracewarden-token-huggingface"))
    api = HfApi(); repo = 'ArsalanKaleem/agentdojo-steps-raw'
    api.create_repo(repo, repo_type='dataset', private=True, exist_ok=True)
    api.upload_folder(folder_path=WORK, repo_id=repo, repo_type='dataset', allow_patterns=list(patterns))
    print("uploaded to", repo)

def restore():
    """After a session restart: pull previously uploaded logs back before converting."""
    from huggingface_hub import snapshot_download
    snapshot_download('ArsalanKaleem/agentdojo-steps-raw', repo_type='dataset',
                      local_dir=f'{WORK}/restore', allow_patterns=[f'logs_{TAG}_*/**'])
    !cp -rn {WORK}/restore/logs_{TAG}_* {WORK}/ 2>/dev/null
    !find {WORK}/logs_{TAG}_ii -name "*.json" 2>/dev/null | wc -l

In [ ]:
# 5 - SMOKE TEST (2 min). Do not skip: this is where a broken tool parser shows up.
!{BASE} -s banking -ut user_task_0 -ut user_task_1 --attack important_instructions --logdir {WORK}/smoke_{TAG}
smoke = convert(f"{WORK}/smoke_{TAG}", f"smoke_{TAG}_labeled", attacked=True)

# expect: >0 trajectories, injection_point on most, some hijacked.
# 0 trajectories = the model never emitted parsable tool calls -> stop and fix the parser.
for t in smoke[:2]:
    print("\n===", t.id, "|", t.label_string, "| security:", t.meta.get("security"))
    for i, s in enumerate(t.steps):
        print(f"  [{i}] {s.label:16s} {s.tool}({str(s.args)[:80]})")

## Benign runs (the negatives)
About 1 hour for three suites. Workspace is excluded: its prompts exceed what a T4 can serve.

In [ ]:
# 6 - benign runs
for s in SUITES:
    print("=" * 30, s)
    !{BASE} -s {s} --logdir {WORK}/logs_{TAG}_benign
_ = convert(f"{WORK}/logs_{TAG}_benign", f"dojo_{TAG}_benign", attacked=False)
upload()

## Attacked runs, one suite per cell
Convert and upload after each one, so a dead session costs at most one suite. If the session restarts, run
cells 1, 3, 4 again and call `restore()` before converting.

In [ ]:
# 7 - banking
!{BASE} -s banking --attack important_instructions --logdir {WORK}/logs_{TAG}_ii
_ = convert(f"{WORK}/logs_{TAG}_ii", f"dojo_{TAG}_ii", attacked=True); upload()

In [ ]:
# 8 - slack  (model A: 54% attack success here, the most vulnerable environment)
!{BASE} -s slack --attack important_instructions --logdir {WORK}/logs_{TAG}_ii
_ = convert(f"{WORK}/logs_{TAG}_ii", f"dojo_{TAG}_ii", attacked=True); upload()

In [ ]:
# 9 - travel  (expect a context-length crash partway; partial coverage is fine, documented)
!{BASE} -s travel --attack important_instructions --logdir {WORK}/logs_{TAG}_ii
_ = convert(f"{WORK}/logs_{TAG}_ii", f"dojo_{TAG}_ii", attacked=True); upload()

In [ ]:
# 10 - freeze a named copy so later runs cannot overwrite model B's data
!cp {WORK}/dojo_{TAG}_ii.jsonl {WORK}/dojo_{TAG}_ii_final.jsonl
!cp {WORK}/dojo_{TAG}_benign.jsonl {WORK}/dojo_{TAG}_benign_final.jsonl
upload()
print("model B complete")

## Before you close the session
1. **Save Version -> Save & Run All** to preserve the notebook and its outputs.
2. On the laptop, pull everything and audit 30 of model B's trajectories:
```
python -c "from huggingface_hub import snapshot_download; snapshot_download('ArsalanKaleem/agentdojo-steps-raw', repo_type='dataset', local_dir='data/agentdojo_raw')"
python scripts/review_labels.py --data data/agentdojo_raw/dojo_qwen14b_ii_final.jsonl --n 30
```
3. Then the transfer table - no GPU needed:
```
tracewarden encode --data data/real_qwen   --encoder <same encoder as training> --out cache/real_qwen
tracewarden encode --data data/real_qwen14b --encoder <same> --out cache/real_qwen14b
python scripts/transfer_eval.py --checkpoint runs/sg.pt \
    --eval agentdrift:data/processed:test:cache/hashing \
    --eval real-A:data/real_qwen:test:cache/real_qwen \
    --eval real-B:data/real_qwen14b:test:cache/real_qwen14b
```